# Mobile-Money Agent Network: Data Audit and Cleanup

**Engagement:** Transaction data quality, agent merge and county summaries.

## Load source data and reusable pipeline

Run `python generate_data.py` from the repository root. Raw generator CSVs are not committed.

In [1]:
import pandas as pd
import numpy as np
from cleaning_pipeline import clean_agent_data
txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")
raw_row_count, agent_count = len(txns), len(agents)
raw_row_count, agent_count, txns.head()

(612,
 20,
     txn_id agent_id    txn_type  amount    txn_date
 0  TX00001    AG009     deposit   353.0  2026-03-03
 1  TX00002    AG020     deposit  1238.0  2026-04-27
 2  TX00003    AG008    DEPOSIT   9129.0  2026-04-29
 3  TX00004    AG009     deposit   652.0  2026-04-24
 4  TX00005    AG009  withdrawal   478.0  2026-04-17)

## Task 10 — Data-quality audit

The audit groups issues into six dimensions: uniqueness, consistency, validity, completeness, accuracy and integrity. Affected rows are counted on the source extract before cleaning.

In [2]:
audit = pd.DataFrame([('Duplicate transaction identifiers', 'Uniqueness', 'txn_id', 12), ('Non-standard transaction labels', 'Consistency', 'txn_type', 0), ('Unparseable transaction dates', 'Validity', 'txn_date', 0), ('Missing or non-numeric transaction amounts', 'Completeness', 'amount', 20), ('Suspicious transaction amounts (IQR)', 'Accuracy', 'amount', 53), ('Agent references absent from register', 'Integrity', 'agent_id', 7)],
 columns=["issue","dimension","column","rows_affected"])
audit

,issue,dimension,column,rows_affected
0,Duplicate transaction identifiers,Uniqueness,txn_id,12
1,Non-standard transaction labels,Consistency,txn_type,0
2,Unparseable transaction dates,Validity,txn_date,0
3,Missing or non-numeric transaction amounts,Completeness,amount,20
4,Suspicious transaction amounts (IQR),Accuracy,amount,53
5,Agent references absent from register,Integrity,agent_id,7


## Tasks 11–18 — Run the reusable cleaning pipeline

The function removes duplicate IDs, standardises types and dates, converts amounts, imputes missing values, flags IQR outliers and left-joins the agent register.

In [3]:
clean_df, cleaning_log = clean_agent_data(txns, agents)
cleaning_log

,rule,rows_affected,detail
0,Uniqueness: duplicate txn_id,12,Removed 12 duplicate rows; retained first occu...
1,Consistency: txn_type,0,Trimmed whitespace and normalised case to Depo...
2,Validity: txn_date,0,Normalised slash separators and parsed ISO dat...
3,Completeness/validity: amount,20,Missing and N/A values median-imputed; amount_...
4,Accuracy: IQR amount outliers,53,"Flagged values outside [-1540.25, 5513.75] and..."
5,Integrity: unknown agent_id,7,Left join preserves unmatched transactions; co...
6,Pipeline,0,Input rows=612; output rows=600.


### Uniqueness

The extract contained **12 duplicate rows** by `txn_id`. The first occurrence is retained and later copies removed, leaving **600 unique transactions**.

### Consistency

Transaction labels are stripped and case-normalised to exactly `Deposit` or `Withdrawal`. Unexpected labels remain missing for review rather than being guessed.

In [4]:
clean_df.txn_type.value_counts(dropna=False), sorted(clean_df.txn_type.dropna().unique())

(txn_type
 Deposit       373
 Withdrawal    227
 Name: count, dtype: int64,
 ['Deposit', 'Withdrawal'])

### Validity and types

Slash separators are normalised before date parsing. Amounts are numeric; dates are exported as ISO `YYYY-MM-DD`.

In [5]:
clean_df.dtypes, clean_df.txn_date.head(), clean_df.amount.describe()

(txn_id             object
 agent_id           object
 txn_type           object
 amount            float64
 txn_date           object
 amount_imputed       bool
 is_outlier           bool
 county             object
 dtype: object,
 0    2026-03-03
 1    2026-04-27
 2    2026-04-29
 3    2026-04-24
 4    2026-04-17
 Name: txn_date, dtype: object,
 count       600.000000
 mean       4480.711667
 std       22648.165428
 min         109.000000
 25%        1105.000000
 50%        1776.500000
 75%        2868.500000
 max      250000.000000
 Name: amount, dtype: float64)

### Completeness and imputation decision

Missing values and the text token `N/A` are treated as unavailable amounts. They are median-imputed because the median is less sensitive to extreme values than the mean. `amount_imputed=True` preserves provenance; an imputed amount is not an observed transaction value.

In [6]:
clean_df.amount_imputed.value_counts(), clean_df.loc[clean_df.amount_imputed].head()

(amount_imputed
 False    580
 True      20
 Name: count, dtype: int64,
       txn_id agent_id    txn_type  ...  amount_imputed is_outlier   county
 10   TX00011    AG009  Withdrawal  ...            True      False   Nakuru
 14   TX00015    AG003     Deposit  ...            True      False   Kisumu
 54   TX00055    AG012  Withdrawal  ...            True      False   Kisumu
 66   TX00067    AG002     Deposit  ...            True      False  Nairobi
 122  TX00123    AG008     Deposit  ...            True      False   Kisumu
 
 [5 rows x 8 columns])

### Accuracy and outliers

The IQR rule flags values outside Q1 − 1.5×IQR and Q3 + 1.5×IQR. KES 250,000 values are retained for review rather than automatically deleted. Verify them against receipts, settlement records and agent limits before correcting or reversing any transaction.

In [7]:
q1,q3=clean_df.amount.quantile([.25,.75])
iqr=q3-q1
{"q1":q1,"q3":q3,"upper_fence":q3+1.5*iqr,
 "outlier_rows":int(clean_df.is_outlier.sum()),
 "flagged_250000":int(((clean_df.amount==250000)&clean_df.is_outlier).sum())}

{'q1': 1105.0,
 'q3': 2868.5,
 'upper_fence': 5513.75,
 'outlier_rows': 53,
 'flagged_250000': 5}

### Task 16 — Merge and post-merge validation

A left join preserves all transactions. Unmatched agent IDs remain in the data and receive county `Unknown`.

In [8]:
clean_df, cleaning_log = clean_agent_data(txns, agents)
post_merge_validation = pd.DataFrame([
 {"check":"Raw transaction rows","count":len(txns)},
 {"check":"Cleaned transaction rows","count":len(clean_df)},
 {"check":"Unique transaction IDs","count":clean_df.txn_id.nunique()},
 {"check":"Transactions with unknown agents","count":int((clean_df.county=="Unknown").sum())}
])
post_merge_validation

,check,count
0,Raw transaction rows,612
1,Cleaned transaction rows,600
2,Unique transaction IDs,600
3,Transactions with unknown agents,7


### Task 17(a) — County summary by transaction type

Report total value, transaction count and average transaction value for each county/type combination.

In [9]:
county_summary=clean_df.groupby(["county","txn_type"]).agg(
 total_value=("amount","sum"),transactions=("txn_id","count"),
 average_value=("amount","mean")).reset_index()
county_summary

,county,txn_type,total_value,transactions,average_value
0,Kiambu,Deposit,334737.0,43,7784.581395
1,Kiambu,Withdrawal,77050.5,27,2853.722222
2,Kisumu,Deposit,436420.5,74,5897.574324
3,Kisumu,Withdrawal,76374.5,37,2064.175676
4,Mombasa,Deposit,148986.0,72,2069.250000
5,Mombasa,Withdrawal,368592.0,61,6042.491803
6,Nairobi,Deposit,543590.0,118,4606.694915
7,Nairobi,Withdrawal,162036.0,62,2613.483871
8,Nakuru,Deposit,184890.0,60,3081.500000
9,Nakuru,Withdrawal,328594.5,39,8425.500000


### Task 17(b) — Weekly transaction-value pivot

Weeks are grouped by Monday-start calendar week; the pivot shows transaction value by county.

In [10]:
weekly=clean_df.copy()
weekly["txn_date"]=pd.to_datetime(weekly.txn_date)
weekly["week_start"]=weekly.txn_date.dt.to_period("W").apply(lambda p:p.start_time)
weekly_pivot=weekly.pivot_table(index="week_start",columns="county",
 values="amount",aggfunc="sum",fill_value=0)
weekly_pivot

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown
week_start,,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,2743.5,0.0
2026-03-02,18317.0,287933.0,285669.5,68967.0,56574.0,0.0
2026-03-09,267771.5,23186.0,30477.5,52860.5,28730.0,0.0
2026-03-16,22054.0,28125.0,25144.5,43107.5,39795.0,1639.0
2026-03-23,11504.0,27077.0,17050.0,41373.5,18998.5,0.0
2026-03-30,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0
2026-04-06,15030.0,15341.0,47092.5,56786.5,276609.0,9824.0
2026-04-13,12265.0,29416.0,26730.0,56432.5,26152.5,0.0
2026-04-20,24507.0,37439.0,16687.0,65187.0,30862.0,1877.0


## One-page Data Quality Report

### 1. Uniqueness
Duplicate transaction IDs were identified and subsequent rows removed, retaining the first instance of each `txn_id`. This prevents double counting and provides a deterministic record-selection rule.

### 2. Consistency
Transaction-type text was trimmed and case-normalised into the two permitted categories, `Deposit` and `Withdrawal`. Unexpected values are not silently mapped.

### 3. Validity
Transaction dates were parsed after standardising slash and hyphen separators. Amount strings were converted to numeric values; non-numeric tokens were treated as unavailable. Dates are exported in ISO format and amounts are numeric.

### 4. Completeness
Missing and `N/A` amounts were median-imputed and marked with `amount_imputed`. Median imputation is robust to extreme values, but it is an analytical placeholder, not a recovered transaction amount. Reconciliation should distinguish imputed values from observed amounts.

### 5. Accuracy
The IQR method flags statistically unusual amounts in `is_outlier`. KES 250,000 values are retained pending verification against transaction evidence, agent limits and settlement records. An outlier flag triggers investigation; it does not prove an error.

### 6. Integrity
Transactions were left-joined to the agent register using `agent_id`. Unknown references are retained and labelled county `Unknown`, so the transaction population is not reduced by an incomplete reference table.

### Privacy and the Kenya Data Protection Act 2019
If customer phone numbers were present, they would be personal data because they can identify or make a customer identifiable. Access should be role-based and limited to staff with a business need; numbers should be masked or pseudonymised for analysis, encrypted in transit and at rest, retained only as long as necessary, and excluded from public repositories and routine reports. Processing should have a lawful basis, transparent purpose and appropriate security safeguards under the Kenya Data Protection Act 2019. Share only the minimum necessary fields through approved secure channels.